# 🌊 SCREENING EXPERIMENT: 10 NOVEL EDGE MAMBA ARCHITECTURES FOR FLOOD SEGMENTATION
**Experimental Protocol:**
- **Models Evaluated:** 10 Novel Mamba Edge Variants (FlowMamba, WaveMamba, MorphoMamba, RippleMamba, AeroMamba, TopoMamba, HydroMamba, DeltaMamba, BoundaryMamba, ResMambaNet)
- **Seed:** Fixed Seed 42 (Strict Deterministic Mode)
- **Input Resolution:** 128x128
- **Datasets:** `floodkaggle` and `aifloodsense`
- **Loss:** Lovasz-Focal Loss (Boundary-Aware Binary Segmentation)
- **Hardware:** NVIDIA Tesla T4 with Official Mamba Hardware CUDA Kernel

In [ ]:
# 1. Clone repository & Install base requirements
!git clone https://github.com/doantrongthai/flood_mamba_edge
%cd flood_mamba_edge
!pip install -r requirements.txt


In [ ]:
# 2. Setup Official Hardware-Accelerated Mamba CUDA Kernel on Google Colab GPU (Tesla T4)
import zipfile, json, sys, sysconfig, pathlib, subprocess, torch

SP = pathlib.Path(sysconfig.get_paths()["purelib"])
FILE_ID = "1kuBAKlNtIc8G1KQFqO8xV7t9ZEM4XATB"

assert torch.cuda.get_device_capability(0) == (7, 5), f"Vui long chon Colab GPU la Tesla T4 (capability 7.5), hien tai: {torch.cuda.get_device_name(0)}"

# 1) Tai goi pre-compiled CUDA wheel
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"])
import gdown
gdown.download(id=FILE_ID, output="/content/pack.zip", quiet=False)

# 2) Giai nen vao site-packages
with zipfile.ZipFile("/content/pack.zip") as z:
    assert z.testzip() is None, "File tai ve bi hong, vui long tai lai"
    m = json.loads(z.read("__manifest__.json"))
    now = {"python": sys.version.split()[0], "torch": torch.__version__, "cuda": torch.version.cuda}
    bad = {k: (m[k], now[k]) for k in now if m[k] != now[k]}
    assert not bad, f"Moi truong Colab khong khop: {bad}"
    for name in z.namelist():
        if not name.startswith("__"):
            z.extract(name, SP)
print("Da giai nen vao:", SP)

# 3) Cai cac dependency bo tro
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "packaging", "ninja", "transformers", "triton"])

# 4) Kiem tra CUDA kernel Mamba tren GPU
import selective_scan_cuda, causal_conv1d_cuda
from mamba_ssm.modules.mamba_simple import causal_conv1d_fn
from mamba_ssm import Mamba
print("causal_conv1d_fn:", "OK (CUDA kernel)" if causal_conv1d_fn is not None else "FALLBACK")
y = Mamba(d_model=64).cuda()(torch.randn(2, 128, 64, device="cuda"))
y.mean().backward()
print("[OK] OFFICIAL MAMBA HARDWARE CUDA KERNEL ACTIVE:", y.shape)


In [ ]:
# 3. Download Datasets Upfront (FloodKaggle & AIFloodSense)
from benchmark import download_dataset

print("--- Downloading FloodKaggle ---")
download_dataset("floodkaggle")
print("--- Downloading AIFloodSense ---")
download_dataset("aifloodsense")
print("[OK] Both datasets downloaded and verified successfully!")


In [ ]:
# 4. Define Candidate Architectures & Screening Configuration
CANDIDATE_MODELS = [
    "flowmamba",      # 1. Directional Strip Mamba (Winding stream/flow modeling)
    "wavemamba",      # 2. Wavelet-Split Mamba (High/Low frequency decomposition)
    "morphomamba",    # 3. Morphology-Gated Mamba (Dilation/Erosion boundary prior)
    "ripplemamba",    # 4. Concentric Multi-Stride Radial Scanning Mamba
    "aeromamba",      # 5. Ultra-light UAV Drone Mamba (<0.1M params, PixelShuffle)
    "topomamba",      # 6. Topographic Gradient-Modulated Mamba
    "hydromamba",     # 7. Dual-Path Local CNN + Global Mamba Hybrid
    "deltamamba",     # 8. Multi-Receptive Field Dilated SSM
    "boundarymamba",  # 9. Boundary Enhancement Module
    "resmambanet"     # 10. Inverted Bottleneck Residual Mamba
]

IMAGE_SIZE = 128
BATCH_SIZE = 4
EPOCHS = 100
LR = 0.001
LOSS_NAME = "lovasz_focal"
NUM_CLASSES = 1
SEED = 42
DATASETS = ["floodkaggle", "aifloodsense"]

print(f"[OK] Ready to screen {len(CANDIDATE_MODELS)} novel architectures on Seed {SEED} across {DATASETS}!")


In [ ]:
# 5. Automated Hardware Profiling Table (Params, FLOPs, Latency, FPS)
import pandas as pd
from models import get_model
from utils.metrics import calculate_model_complexity, measure_inference_time

profiling_records = []
print("="*75)
print("HARDWARE & COMPLEXITY PROFILING ACROSS 10 NOVEL ARCHITECTURES")
print("="*75)

for model_name in CANDIDATE_MODELS:
    model = get_model(model_name, num_classes=NUM_CLASSES, input_size=IMAGE_SIZE, seed=SEED)
    complexity = calculate_model_complexity(model, input_size=IMAGE_SIZE)
    perf = measure_inference_time(model, input_size=IMAGE_SIZE, num_iterations=100)
    
    record = {
        "Model": model_name,
        "Params": complexity["deployable_params"],
        "Size (MB)": complexity["model_size_mb"],
        "GFLOPs": complexity["gflops"],
        "Latency (ms)": perf["mean_time_ms"],
        "FPS": perf["fps"]
    }
    profiling_records.append(record)
    print(f"{model_name:<16} | Params: {record['Params']:>8,d} | Size: {record['Size (MB)']:>5.2f} MB | GFLOPs: {record['GFLOPs']:>6.4f} | Latency: {record['Latency (ms)']:>6.2f} ms | FPS: {record['FPS']:>6.1f}")

df_profile = pd.DataFrame(profiling_records)
display(df_profile)


In [ ]:
# 6. SCREENING EXECUTION LOOP: Train each model on FloodKaggle & AIFloodSense (Seed 42)
import os, re, subprocess

screening_results = {}

for idx, model_name in enumerate(CANDIDATE_MODELS, 1):
    print(f"\n{'#'*75}")
    print(f"[{idx}/{len(CANDIDATE_MODELS)}] SCREENING ARCHITECTURE: {model_name.upper()}")
    print(f"{'#'*75}")
    screening_results[model_name] = {}
    
    for ds in DATASETS:
        print(f"\n>>> Training {model_name} on {ds.upper()} (Seed {SEED}) <<<")
        cmd = [
            sys.executable, "main.py",
            "--dataset", ds,
            "--model", model_name,
            "--num_classes", str(NUM_CLASSES),
            "--loss", LOSS_NAME,
            "--size", str(IMAGE_SIZE),
            "--epochs", str(EPOCHS),
            "--batch_size", str(BATCH_SIZE),
            "--lr", str(LR),
            "--seed", str(SEED)
        ]
        proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        output = proc.stdout
        print("\n".join(output.splitlines()[-25:]))  # In 25 dong ket qua cuoi
        
        # Parse metrics
        miou_match = re.search(r"mIoU:\s+([0-9.]+)", output)
        dice_match = re.search(r"Dice / F1 Score:\s+([0-9.]+)", output)
        biou_match = re.search(r"Boundary IoU:\s+([0-9.]+)", output)
        
        miou_val = float(miou_match.group(1)) if miou_match else None
        dice_val = float(dice_match.group(1)) if dice_match else None
        biou_val = float(biou_match.group(1)) if biou_match else None
        
        screening_results[model_name][ds] = {
            "mIoU": miou_val,
            "Dice": dice_val,
            "Boundary_IoU": biou_val
        }
        print(f"[DONE] {model_name} on {ds}: mIoU={miou_val}, Dice={dice_val}, Boundary_IoU={biou_val}")


In [ ]:
# 7. SUMMARY LEADERBOARD & COMPARATIVE RANKING
import pandas as pd

summary_rows = []
for m in CANDIDATE_MODELS:
    fk = screening_results.get(m, {}).get("floodkaggle", {})
    afs = screening_results.get(m, {}).get("aifloodsense", {})
    
    fk_miou = fk.get("mIoU") or 0.0
    afs_miou = afs.get("mIoU") or 0.0
    mean_miou = (fk_miou + afs_miou) / 2.0
    
    # Profile data
    p = df_profile[df_profile["Model"] == m].iloc[0]
    
    summary_rows.append({
        "Architecture": m,
        "Params": p["Params"],
        "GFLOPs": p["GFLOPs"],
        "Latency (ms)": p["Latency (ms)"],
        "FPS": p["FPS"],
        "FloodKaggle mIoU": fk_miou,
        "AIFloodSense mIoU": afs_miou,
        "Mean mIoU": mean_miou
    })

df_summary = pd.DataFrame(summary_rows).sort_values(by="Mean mIoU", ascending=False).reset_index(drop=True)
df_summary.index = df_summary.index + 1
df_summary.index.name = "Rank"

print("="*85)
print("🏆 FINAL LEADERBOARD: 10 NOVEL EDGE MAMBA FLOOD SEGMENTATION ARCHITECTURES")
print("="*85)
display(df_summary)

# Save to CSV
df_summary.to_csv("outputs/mamba_screening_leaderboard.csv")
print("[OK] Leaderboard saved to outputs/mamba_screening_leaderboard.csv!")


In [ ]:
# 8. Bundle All Checkpoints & Download Results Package
import os, shutil
from google.colab import files

bundle_dir = "Mamba_Screening_10_Artifacts"
os.makedirs(bundle_dir, exist_ok=True)
output_dir = "outputs"

if os.path.exists(output_dir):
    for item in sorted(os.listdir(output_dir)):
        src = os.path.join(output_dir, item)
        dst = os.path.join(bundle_dir, item)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        elif os.path.isfile(src):
            shutil.copy2(src, dst)

zip_filename = "Mamba_Screening_10_Artifacts.zip"
if os.path.exists(zip_filename):
    os.remove(zip_filename)

shutil.make_archive("Mamba_Screening_10_Artifacts", "zip", bundle_dir)
print(f"[OK] Artifacts compressed successfully: {zip_filename}")
files.download(zip_filename)
